# Exp05.基于Transformer的极简翻译任务（Pytorch）

本次实践没有用大型的数据集，而是手动输入了两对中文→英语的句子

构建词库的时候，每个字的索引也是手动硬编码的，主要是为了降低代码阅读难度

## 1. 数据构建

In [ ]:
import math
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as Data

device = 'cpu'
# device = 'cuda'

# transformer epochs
epochs = 100



In [ ]:
sentences = [
    # 中文和英语的单词个数不要求相同
    # enc_input                dec_input           dec_output
    ['我 想要 一杯 啤酒 P', 'S i want a beer .', 'i want a beer . E'],
    ['我 想要 一杯 可乐 P', 'S i want a coke .', 'i want a coke . E']
]

# 中文和英语的单词要分开建立词库
# Padding Should be Zero
src_vocab = {'P': 0, '我': 1, '想要': 2, '一杯': 3, '啤酒': 4, '可乐': 5}
src_idx2word = {i: w for i, w in enumerate(src_vocab)}
print("src_idx2word:",src_idx2word)
src_vocab_size = len(src_vocab)

tgt_vocab = {'P': 0, 'i': 1, 'want': 2, 'a': 3, 'beer': 4, 'coke': 5, 'S': 6, 'E': 7, '.': 8}
idx2word = {i: w for i, w in enumerate(tgt_vocab)}
print("idx2word:",idx2word)
tgt_vocab_size = len(tgt_vocab)

src_len = 5  # （源句子的长度）enc_input max sequence length
tgt_len = 6  # dec_input(=dec_output) max sequence length

# Transformer Parameters
d_model = 512  # Embedding Size（token embedding和position编码的维度）
d_ff = 2048  # FeedForward dimension (两次线性层中的隐藏层 512->2048->512，线性层是用来做特征提取的），当然最后会再接一个projection层
d_k = d_v = 64  # dimension of K(=Q), V（Q和K的维度需要相同，这里为了方便让K=V）
n_layers = 6  # number of Encoder of Decoder Layer（Block的个数）
n_heads = 8  # number of heads in Multi-Head Attention（有几套头）

In [ ]:
def make_data(sentences):
    """把单词序列转换为数字序列"""
    enc_inputs, dec_inputs, dec_outputs = [], [], []
    for i in range(len(sentences)):
        enc_input = [[src_vocab[n] for n in sentences[i][0].split()]]  # [[1, 2, 3, 4, 0], [1, 2, 3, 5, 0]]
        dec_input = [[tgt_vocab[n] for n in sentences[i][1].split()]]  # [[6, 1, 2, 3, 4, 8], [6, 1, 2, 3, 5, 8]]
        dec_output = [[tgt_vocab[n] for n in sentences[i][2].split()]]  # [[1, 2, 3, 4, 8, 7], [1, 2, 3, 5, 8, 7]]

        enc_inputs.extend(enc_input)
        dec_inputs.extend(dec_input)
        dec_outputs.extend(dec_output)
    # 将列表转换为 PyTorch 的 LongTensor（适合用于 embedding lookup）
    return torch.LongTensor(enc_inputs), torch.LongTensor(dec_inputs), torch.LongTensor(dec_outputs)


enc_inputs, dec_inputs, dec_outputs = make_data(sentences)

In [ ]:
class MyDataSet(Data.Dataset):
    """自定义DataLoader"""

    def __init__(self, enc_inputs, dec_inputs, dec_outputs):
        super(MyDataSet, self).__init__()
        self.enc_inputs = enc_inputs
        self.dec_inputs = dec_inputs
        self.dec_outputs = dec_outputs

    def __len__(self):
        return self.enc_inputs.shape[0]

    def __getitem__(self, idx):
        return self.enc_inputs[idx], self.dec_inputs[idx], self.dec_outputs[idx]


loader = Data.DataLoader(MyDataSet(enc_inputs, dec_inputs, dec_outputs), 2, True)

## 2. Transformer模型定义

In [ ]:
# (3)位置编码层：为模型添加位置信息。
class PositionalEncoding(nn.Module):
    """
    位置编码通过增加模型对序列中元素顺序的敏感性，帮助模型理解输入序列的位置关系。
    这是通过将一个固定模式的编码添加到输入序列的每个位置来实现的。

    参数:
        d_model: 模型的维度，即输入张量的特征维度。
        dropout: Dropout层的失活率，默认为0.1。
        max_len: 预定义的最大序列长度，默认为5000。
    """
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        ## 位置编码的实现其实很简单，直接对照着公式去敲代码就可以，下面这个代码只是其中一种实现方式；
        ## 从理解来讲，需要注意的就是偶数和奇数在公式上有一个共同部分，我们使用log函数把次方拿下来，方便计算；
        ## pos代表的是单词在句子中的索引，这点需要注意；比如max_len是128个，那么索引就是从0，1，2，...,127
        ##假设我的d_model是512，2i那个符号中i从0取到了255，那么2i对应取值就是0,2,4...510
        
        # 初始化位置编码张量
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        前向传播函数，对输入序列添加位置编码。

        参数:
            x: 输入序列张量，形状为[seq_len, batch_size, d_model]。

        返回:
            添加了位置编码的序列张量，经过dropout处理。
        """
        # 将位置编码添加到输入序列
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)

# (4) get_attn_pad_mask 
def get_attn_pad_mask(seq_q, seq_k):
    # pad mask的作用：在对value向量加权平均的时候，可以让pad对应的alpha_ij=0，这样注意力就不会考虑到pad向量
    """这里的q,k表示的是两个序列（跟注意力机制的q,k没有关系），例如encoder_inputs (x1,x2,..xm)和encoder_inputs (x1,x2..xm)
    encoder和decoder都可能调用这个函数，所以seq_len视情况而定
    seq_q: [batch_size, seq_len]
    seq_k: [batch_size, seq_len]
    seq_len could be src_len or it could be tgt_len
    seq_len in seq_q and seq_len in seq_k maybe not equal
    """
    batch_size, len_q = seq_q.size()  # 这个seq_q只是用来expand维度的
    batch_size, len_k = seq_k.size()
    # eq(zero) is PAD token
    # 例如:seq_k = [[1,2,3,4,0], [1,2,3,5,0]]
    pad_attn_mask = seq_k.data.eq(0).unsqueeze(1)  # [batch_size, 1, len_k], True is masked
    return pad_attn_mask.expand(batch_size, len_q, len_k)  # [batch_size, len_q, len_k] 构成一个立方体(batch_size个这样的矩阵)

def get_attn_subsequence_mask(seq):
    """
    生成后续注意力掩码，防止模型在解码时关注到当前位置之后的 token。

    seq: [batch_size, tgt_len]
        输入序列张量，形状为 [batch_size, tgt_len]，其中 batch_size 是批量大小，tgt_len 是目标序列长度。
    """
    # 根据输入序列的形状计算注意力掩码的形状，即 [batch_size, tgt_len, tgt_len]。
    attn_shape = [seq.size(0), seq.size(1), seq.size(1)]
    # attn_shape: [batch_size, tgt_len, tgt_len]
    # 使用 numpy 生成一个上三角矩阵，用于屏蔽后续的 token。
    subsequence_mask = np.triu(np.ones(attn_shape), k=1)  # 生成一个上三角矩阵
    # 将生成的掩码转换为 PyTorch 张量，并确保其类型为 byte，以便用于后续的掩码操作。
    subsequence_mask = torch.from_numpy(subsequence_mask).byte()
    # 返回生成的后续注意力掩码张量，其形状为 [batch_size, tgt_len, tgt_len]。
    return subsequence_mask  # [batch_size, tgt_len, tgt_len]

In [ ]:
# (7)缩放点积注意力
class ScaledDotProductAttention(nn.Module):
    """
    缩放点积注意力机制（Scaled Dot-Product Attention）的实现。
    该机制是Transformer模型中关键组件之一，负责计算查询（Query）、键（Key）和值（Value）之间的注意力权重，
    并输出加权后的上下文向量。此过程通过矩阵乘法实现，提高了计算效率和并行性。
    """
    def __init__(self):
        super(ScaledDotProductAttention, self).__init__()

    def forward(self, Q, K, V, attn_mask):
        """
        定义前向传播过程，计算缩放点积注意力。

        参数:
        Q: 查询矩阵，形状为 [batch_size, n_heads, len_q, d_k]。
        K: 键矩阵，形状为 [batch_size, n_heads, len_k, d_k]。
        V: 值矩阵，形状为 [batch_size, n_heads, len_v(=len_k), d_v]。
        attn_mask: 注意力掩码矩阵，形状为 [batch_size, n_heads, seq_len, seq_len]。
            说明：在encoder-decoder的Attention层中len_q(q1,..qt)和len_k(k1,...km)可能不同。

        返回:
        context: 上下文矩阵，形状为 [batch_size, n_heads, len_q, d_v]。
        attn: 注意力权重矩阵，用于可视化等目的，形状为 [batch_size, n_heads, len_q, len_k]。
        """
        scores = torch.matmul(Q, K.transpose(-1, -2)) / np.sqrt(d_k)  # 计算点积并缩放，scores : [batch_size, n_heads, len_q, len_k]
        # mask矩阵填充scores（用-1e9填充scores中与attn_mask中值为1位置相对应的元素）
        scores.masked_fill_(attn_mask, -1e9)  # 对注意力掩码应用填充操作

        attn = nn.Softmax(dim=-1)(scores)  # 对最后一个维度(v)做softmax，得到注意力权重
        # scores : [batch_size, n_heads, len_q, len_k] * V: [batch_size, n_heads, len_v(=len_k), d_v]
        context = torch.matmul(attn, V)  # 通过注意力权重对值矩阵进行加权求和，得到上下文矩阵
        # context：[[z1,z2,...],[...]]向量, attn注意力稀疏矩阵（用于可视化的）
        return context, attn


# (6)多头注意力
class MultiHeadAttention(nn.Module):
    """这个Attention类可以实现:
    Encoder的Self-Attention
    Decoder的Masked Self-Attention
    Encoder-Decoder的Attention
    """

    def __init__(self):
        super(MultiHeadAttention, self).__init__()
        self.W_Q = nn.Linear(d_model, d_k * n_heads, bias=False)  # q,k必须维度相同，不然无法做点积
        self.W_K = nn.Linear(d_model, d_k * n_heads, bias=False)
        self.W_V = nn.Linear(d_model, d_v * n_heads, bias=False)
        self.fc = nn.Linear(n_heads * d_v, d_model, bias=False)

    def forward(self, input_Q, input_K, input_V, attn_mask):
        """
        input_Q: [batch_size, len_q, d_model]
        input_K: [batch_size, len_k, d_model]
        input_V: [batch_size, len_v(=len_k), d_model]
        attn_mask: [batch_size, seq_len, seq_len]
        """
        residual, batch_size = input_Q, input_Q.size(0)
        # 下面的多头的参数矩阵是放在一起做线性变换的，然后再拆成多个头，这是工程实现的技巧
        # B: batch_size, S:seq_len, D: dim
        # (B, S, D) -proj-> (B, S, D_new) -split-> (B, S, Head, W) -trans-> (B, Head, S, W)
        #           线性变换               拆成多头

        #下面这个就是先映射，后分头；一定要注意的是q和k分头之后维度是一致额，所以一看这里都是d_k
        # Q: [batch_size, n_heads, len_q, d_k]
        Q = self.W_Q(input_Q).view(batch_size, -1, n_heads, d_k).transpose(1, 2)
        # K: [batch_size, n_heads, len_k, d_k] # K和V的长度一定相同，维度可以不同
        K = self.W_K(input_K).view(batch_size, -1, n_heads, d_k).transpose(1, 2)
        # V: [batch_size, n_heads, len_v(=len_k), d_v]
        V = self.W_V(input_V).view(batch_size, -1, n_heads, d_v).transpose(1, 2)

        # 因为是多头，所以mask矩阵要扩充成4维的
        # attn_mask: [batch_size, seq_len, seq_len] -> [batch_size, n_heads, seq_len, seq_len]
        attn_mask = attn_mask.unsqueeze(1).repeat(1, n_heads, 1, 1)

        # context: [batch_size, n_heads, len_q, d_v], attn: [batch_size, n_heads, len_q, len_k]
        context, attn = ScaledDotProductAttention()(Q, K, V, attn_mask)
        # 下面将不同头的输出向量拼接在一起
        # context: [batch_size, n_heads, len_q, d_v] -> [batch_size, len_q, n_heads * d_v]
        context = context.transpose(1, 2).reshape(batch_size, -1, n_heads * d_v)
        # 再做一个projection
        output = self.fc(context)  # [batch_size, len_q, d_model]
        return nn.LayerNorm(d_model).to(device)(output + residual), attn


# (8) 前馈神经网络：Pytorch中的Linear只会对最后一维操作，所以正好是我们希望的每个位置用同一个全连接网络
class PoswiseFeedForwardNet(nn.Module):
    """
    实现一个位置前馈神经网络（Poswise Feed Forward Network）。
    该网络结构是Transformer模型中的一个关键组件，用于对输入进行非线性变换。
    它由两个线性层（全连接层）组成，中间夹着一个ReLU激活函数。
    """

    def __init__(self):
        """
        初始化网络结构。使用Sequential容器构建一个前馈神经网络，包括两个线性层和一个ReLU激活函数。
        第一个线性层将输入的维度从d_model扩展到d_ff，第二个线性层将维度从d_ff压缩回d_model。
        这样的设计旨在通过增加非线性来增强模型的表达能力，同时不改变输入输出的维度。
        """
        super(PoswiseFeedForwardNet, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(d_model, d_ff, bias=False),
            nn.ReLU(),
            nn.Linear(d_ff, d_model, bias=False)
        )

    def forward(self, inputs):
        """
        定义网络的前向传播过程。
        
        参数:
        inputs: 输入张量，形状为[batch_size, seq_len, d_model]，代表一批序列数据。
        
        返回:
        输出张量，经过前馈神经网络和残差连接、层归一化处理后得到，形状与输入相同。
        """
        # 保存输入作为残差连接的一部分
        residual = inputs
        # 将输入通过前馈神经网络进行变换
        output = self.fc(inputs)
        # 将网络输出与原始输入相加，实现残差连接，然后通过层归一化处理，保持输出的稳定性
        return nn.LayerNorm(d_model).to(device)(output + residual)  # [batch_size, seq_len, d_model]


# (5)编码器层
class EncoderLayer(nn.Module):
    def __init__(self):
        super(EncoderLayer, self).__init__()
        self.enc_self_attn = MultiHeadAttention()
        self.pos_ffn = PoswiseFeedForwardNet()

    def forward(self, enc_inputs, enc_self_attn_mask):
        """E
        enc_inputs: [batch_size, src_len, d_model]
        enc_self_attn_mask: [batch_size, src_len, src_len]  mask矩阵(pad mask or sequence mask)
        """
        # enc_outputs: [batch_size, src_len, d_model], attn: [batch_size, n_heads, src_len, src_len]
        # 第一个enc_inputs * W_Q = Q
        # 第二个enc_inputs * W_K = K
        # 第三个enc_inputs * W_V = V
        enc_outputs, attn = self.enc_self_attn(enc_inputs, enc_inputs, enc_inputs,
                                               enc_self_attn_mask)  # enc_inputs to same Q,K,V（未线性变换前）
        enc_outputs = self.pos_ffn(enc_outputs)
        # enc_outputs: [batch_size, src_len, d_model]
        return enc_outputs, attn

# (2)编码器Encoder 部分包含三个部分：词向量embedding，位置编码部分，注意力层及后续的前馈神经网络
class Encoder(nn.Module):
    """
    Encoder类是Transformer模型的编码器部分。
    它由一个嵌入层、一个位置编码层和一系列编码器层组成。
    """
    def __init__(self):
        """
        初始化Encoder类。

        参数:
        - src_vocab_size: 源语言词汇表的大小
        - d_model: 模型维度
        - n_layers: 编码器层的数量
        """
        super(Encoder, self).__init__()
        self.src_emb = nn.Embedding(src_vocab_size, d_model)  # token Embedding
        self.pos_emb = PositionalEncoding(d_model)  # Transformer中位置编码时固定的，不需要学习
        self.layers = nn.ModuleList([EncoderLayer() for _ in range(n_layers)])  # 如果你的模块数量是动态的（比如根据配置决定层数），ModuleList 是标准做法

    def forward(self, enc_inputs):
        """
        Encoder的前向传播方法。

        参数:
        - enc_inputs: 编码器的输入序列，形状为[batch_size, src_len]

        返回:
        - enc_outputs: 编码器的输出，形状为[batch_size, src_len, d_model]
        - enc_self_attns: 所有编码器层的自注意力权重列表
        """
        # 应用嵌入层和位置编码层
        enc_outputs = self.src_emb(enc_inputs)  # [batch_size, src_len, d_model]
        enc_outputs = self.pos_emb(enc_outputs.transpose(0, 1)).transpose(0, 1)  # [batch_size, src_len, d_model]

        # 生成编码器输入序列的pad mask矩阵
        enc_self_attn_mask = get_attn_pad_mask(enc_inputs, enc_inputs)  # [batch_size, src_len, src_len]
        enc_self_attns = []  # 保存每层的自注意力权重，用于可视化

        # 通过每一层的编码器层
        for layer in self.layers:
            enc_outputs, enc_self_attn = layer(enc_outputs, enc_self_attn_mask)
            enc_self_attns.append(enc_self_attn)

        return enc_outputs, enc_self_attns


In [ ]:
# (10)解码器层EncoderLayer ：包含多头注意力机制和前馈神经网络
class DecoderLayer(nn.Module):
    """
    解码器层，包含自我注意力机制、编码器-解码器注意力机制和位置前馈网络。
    """
    def __init__(self):
        super(DecoderLayer, self).__init__()
        # 初始化多头自我注意力机制
        self.dec_self_attn = MultiHeadAttention()
        # 初始化多头编码器-解码器注意力机制
        self.dec_enc_attn = MultiHeadAttention()
        # 初始化位置前馈网络
        self.pos_ffn = PoswiseFeedForwardNet()

    def forward(self, dec_inputs, enc_outputs, dec_self_attn_mask, dec_enc_attn_mask):
        """
        前向传播函数

        参数:
        dec_inputs: 解码器输入序列，形状为 [batch_size, tgt_len, d_model]
        enc_outputs: 编码器输出序列，形状为 [batch_size, src_len, d_model]
        dec_self_attn_mask: 解码器自我注意力掩码，形状为 [batch_size, tgt_len, tgt_len]
        dec_enc_attn_mask: 解码器-编码器注意力掩码，形状为 [batch_size, tgt_len, src_len]

        返回值:
        dec_outputs: 解码器输出序列，形状为 [batch_size, tgt_len, d_model]
        dec_self_attn: 解码器自我注意力权重，形状为 [batch_size, n_heads, tgt_len, tgt_len]
        dec_enc_attn: 解码器-编码器注意力权重，形状为 [batch_size, n_heads, tgt_len, src_len]
        """
        # 通过自我注意力机制处理解码器输入
        # dec_outputs: [batch_size, tgt_len, d_model], dec_self_attn: [batch_size, n_heads, tgt_len, tgt_len]
        dec_outputs, dec_self_attn = self.dec_self_attn(dec_inputs, dec_inputs, dec_inputs,
                                                        dec_self_attn_mask)  # 这里Q,K,V全是Decoder自己的输入
        # 通过编码器-解码器注意力机制处理解码器自我注意力输出和编码器输出
        # dec_outputs: [batch_size, tgt_len, d_model], dec_enc_attn: [batch_size, n_heads, tgt_len, src_len]
        dec_outputs, dec_enc_attn = self.dec_enc_attn(dec_outputs, enc_outputs, enc_outputs,
                                                      dec_enc_attn_mask)  # Attention层的Q(来自decoder) 和 K,V(来自encoder)
        # 通过位置前馈网络处理注意力机制的输出
        dec_outputs = self.pos_ffn(dec_outputs)  # [batch_size, tgt_len, d_model]
        # 返回解码器输出序列和注意力权重，这些可以用于可视化或进一步处理
        return dec_outputs, dec_self_attn, dec_enc_attn
    

# (9)解码器
class Decoder(nn.Module):
    """
    Transformer解码器，包含多层带有自注意力机制和编码器-解码器注意力机制的解码器层。
    """
    def __init__(self):
        """
        初始化解码器，包括目标词汇嵌入表、位置编码以及解码器层堆栈。
        """
        super(Decoder, self).__init__()
        self.tgt_emb = nn.Embedding(tgt_vocab_size, d_model)  # 目标序列的嵌入层
        self.pos_emb = PositionalEncoding(d_model)  # 位置编码以向嵌入添加位置信息
        self.layers = nn.ModuleList([DecoderLayer() for _ in range(n_layers)])  # 解码器层列表

    def forward(self, dec_inputs, enc_inputs, enc_outputs):
        """
        前向传播函数。

        参数:
        dec_inputs -- 解码器输入张量，形状为[batch_size, tgt_len]
        enc_inputs -- 编码器输入张量，形状为[batch_size, src_len]
        enc_outputs -- 编码器输出张量，用于编码器-解码器注意力层，形状为[batch_size, src_len, d_model]

        返回:
        dec_outputs -- 解码器输出张量，形状为[batch_size, tgt_len, d_model]
        dec_self_attns -- 自注意力权重列表
        dec_enc_attns -- 编码器-解码器注意力权重列表
        """
        dec_outputs = self.tgt_emb(dec_inputs)  # [batch_size, tgt_len, d_model]
        dec_outputs = self.pos_emb(dec_outputs.transpose(0, 1)).transpose(0, 1).to(device)  # 添加位置编码

        dec_self_attn_pad_mask = get_attn_pad_mask(dec_inputs, dec_inputs).to(device)  # 构建pad掩码矩阵
        dec_self_attn_subsequence_mask = get_attn_subsequence_mask(dec_inputs).to(device)  # 构建subsequence掩码矩阵

        dec_self_attn_mask = torch.gt((dec_self_attn_pad_mask + dec_self_attn_subsequence_mask), 0).to(device)  # 合并两种掩码

        dec_enc_attn_mask = get_attn_pad_mask(dec_inputs, enc_inputs)  # 构建编码器-解码器注意力掩码矩阵

        dec_self_attns, dec_enc_attns = [], []
        for layer in self.layers:
            dec_outputs, dec_self_attn, dec_enc_attn = layer(dec_outputs, enc_outputs, dec_self_attn_mask, dec_enc_attn_mask)
            dec_self_attns.append(dec_self_attn)
            dec_enc_attns.append(dec_enc_attn)

        return dec_outputs, dec_self_attns, dec_enc_attns

In [ ]:
# (1)Transformer模型定义，从整体网路结构来看，分为三个部分：编码层，解码层，输出层
class Transformer(nn.Module):
    def __init__(self):
        super(Transformer, self).__init__()
        self.encoder = Encoder().to(device)
        self.decoder = Decoder().to(device)
        # 输出层 d_model 是我们解码层每个token输出的维度大小，之后会做一个 tgt_vocab_size 大小的softmax
        self.projection = nn.Linear(d_model, tgt_vocab_size, bias=False).to(device)

    def forward(self, enc_inputs, dec_inputs):
        """Transformers的输入：两个序列
        enc_inputs: [batch_size, src_len]，编码端的输入
        dec_inputs: [batch_size, tgt_len]，解码端的输入
        """
        # tensor to store decoder outputs
        # outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(self.device)

        # enc_outputs: [batch_size, src_len, d_model], enc_self_attns: [n_layers, batch_size, n_heads, src_len, src_len]
        # 经过Encoder网络后，得到的输出还是[batch_size, src_len, d_model]，enc_self_attns是QK转置相乘之后softmax之后的矩阵值，代表的是每个单词和其他单词相关性
        enc_outputs, enc_self_attns = self.encoder(enc_inputs)
        
        # dec_outputs: [batch_size, tgt_len, d_model], dec_self_attns: [n_layers, batch_size, n_heads, tgt_len, tgt_len], dec_enc_attn: [n_layers, batch_size, tgt_len, src_len]
        # dec_outputs 是decoder主要输出，用于后续的linear映射； dec_self_attns是查看每个单词对decoder中输入的其余单词的相关性；dec_enc_attns是decoder中每个单词对encoder中每个单词的相关性；
        dec_outputs, dec_self_attns, dec_enc_attns = self.decoder(dec_inputs, enc_inputs, enc_outputs)
        
        # dec_outputs: [batch_size, tgt_len, d_model] -> dec_logits: [batch_size, tgt_len, tgt_vocab_size]
        # dec_outputs做映射到词表大小
        dec_logits = self.projection(dec_outputs)
        return dec_logits.view(-1, dec_logits.size(-1)), enc_self_attns, dec_self_attns, dec_enc_attns

## 3. 模型训练

In [ ]:
model = Transformer().to(device)
# 这里的损失函数里面设置了一个参数 ignore_index=0，因为 "pad" 这个单词的索引为 0，这样设置以后，就不会计算 "pad" 的损失（因为本来 "pad" 也没有意义，不需要计算）
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.SGD(model.parameters(), lr=1e-3, momentum=0.99)  # 用adam的话效果不好

# ====================================================================================================
for epoch in range(epochs):
    for enc_inputs, dec_inputs, dec_outputs in loader:
        """
        enc_inputs: [batch_size, src_len]
        dec_inputs: [batch_size, tgt_len]
        dec_outputs: [batch_size, tgt_len]
        """
        enc_inputs, dec_inputs, dec_outputs = enc_inputs.to(device), dec_inputs.to(device), dec_outputs.to(device)
        # outputs: [batch_size * tgt_len, tgt_vocab_size]
        outputs, enc_self_attns, dec_self_attns, dec_enc_attns = model(enc_inputs, dec_inputs)
        loss = criterion(outputs, dec_outputs.view(-1))  # dec_outputs.view(-1):[batch_size * tgt_len * tgt_vocab_size]
        print('Epoch:', '%04d' % (epoch + 1), 'loss =', '{:.6f}'.format(loss))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


## 4. 模型预测

In [ ]:
def greedy_decoder(model, enc_input, start_symbol):
    """贪心编码
    For simplicity, a Greedy Decoder is Beam search when K=1. This is necessary for inference as we don't know the
    target sequence input. Therefore we try to generate the target input word by word, then feed it into the transformer.
    Starting Reference: http://nlp.seas.harvard.edu/2018/04/03/attention.html#greedy-decoding
    :param model: Transformer Model
    :param enc_input: The encoder input
    :param start_symbol: The start symbol. In this example it is 'S' which corresponds to index 4
    :return: The target input
    """
    enc_outputs, enc_self_attns = model.encoder(enc_input)  # 通过模型的编码器生成编码器输出。
    dec_input = torch.zeros(1, 0).type_as(enc_input.data)  # 初始化解码器输入为一个空张量。
    terminal = False  # 设置终止标志为False。
    next_symbol = start_symbol  # 设置初始符号为起始符号。
    while not terminal:   # 解码循环，直到遇到终止符号。
        # 预测阶段：dec_input序列会一点点变长（每次添加一个新预测出来的单词），在每次循环中，将当前预测的符号next_symbol添加到dec_input中
        dec_input = torch.cat([dec_input.to(device), torch.tensor([[next_symbol]], dtype=enc_input.dtype).to(device)],
                              -1)
        dec_outputs, _, _ = model.decoder(dec_input, enc_input, enc_outputs)  # 通过模型的解码器生成解码器输出dec_outputs。
        projected = model.projection(dec_outputs)  # 通过投影层model.projection(dec_outputs)生成概率分布。
        prob = projected.squeeze(0).max(dim=-1, keepdim=False)[1]
        # 增量更新（我们希望重复单词预测结果是一样的）
        # 我们在预测是会选择性忽略重复的预测的词，只摘取最新预测的单词拼接到输入序列中
        next_word = prob.data[-1]  # 选择概率最高的词作为下一个预测的词next_word。拿出当前预测的单词(数字)。我们用x'_t对应的输出z_t去预测下一个单词的概率，不用z_1,z_2..z_{t-1}
        next_symbol = next_word  # 更新next_symbol为next_word。
        if next_symbol == tgt_vocab["E"]:  # 如果next_symbol是终止符E，则设置terminal为True，结束循环。
            terminal = True
        # print(next_word)

    # greedy_dec_predict = torch.cat(
    #     [dec_input.to(device), torch.tensor([[next_symbol]], dtype=enc_input.dtype).to(device)],
    #     -1)
    greedy_dec_predict = dec_input[:, 1:]  # 返回去掉起始符号后的解码器输入。
    return greedy_dec_predict  # 返回最终的解码结果。


# ==========================================================================================
# 预测阶段
# 从数据加载器中获取一批编码输入数据
enc_inputs, _, _ = next(iter(loader))

# 遍历每个编码输入数据
for i in range(len(enc_inputs)):
    # 使用贪婪解码器对当前编码输入进行预测
    # 这里将单个输入数据 reshape 为 (1, -1) 形式并移动到指定设备上
    greedy_dec_predict = greedy_decoder(model, enc_inputs[i].view(1, -1).to(device), start_symbol=tgt_vocab["S"])

    # 打印原始编码输入数据及其对应的预测结果
    print(enc_inputs[i], '->', greedy_dec_predict.squeeze())

    # 打印编码输入数据的词汇形式及其对应的预测结果的词汇形式
    # 这里将索引转换回词汇以便更好地理解输入和输出
    print([src_idx2word[t.item()] for t in enc_inputs[i]], '->',
          [idx2word[n.item()] for n in greedy_dec_predict.squeeze()])